In [1]:
# Import Dependencies
import yfinance
import pandas_datareader as pdr
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

In [2]:
pdr.famafrench.get_available_datasets()

['F-F_Research_Data_Factors',
 'F-F_Research_Data_Factors_weekly',
 'F-F_Research_Data_Factors_daily',
 'F-F_Research_Data_5_Factors_2x3',
 'F-F_Research_Data_5_Factors_2x3_daily',
 'Portfolios_Formed_on_ME',
 'Portfolios_Formed_on_ME_Wout_Div',
 'Portfolios_Formed_on_ME_Daily',
 'Portfolios_Formed_on_BE-ME',
 'Portfolios_Formed_on_BE-ME_Wout_Div',
 'Portfolios_Formed_on_BE-ME_Daily',
 'Portfolios_Formed_on_OP',
 'Portfolios_Formed_on_OP_Wout_Div',
 'Portfolios_Formed_on_OP_Daily',
 'Portfolios_Formed_on_INV',
 'Portfolios_Formed_on_INV_Wout_Div',
 'Portfolios_Formed_on_INV_Daily',
 '6_Portfolios_2x3',
 '6_Portfolios_2x3_Wout_Div',
 '6_Portfolios_2x3_weekly',
 '6_Portfolios_2x3_daily',
 '25_Portfolios_5x5',
 '25_Portfolios_5x5_Wout_Div',
 '25_Portfolios_5x5_Daily',
 '100_Portfolios_10x10',
 '100_Portfolios_10x10_Wout_Div',
 '100_Portfolios_10x10_Daily',
 '6_Portfolios_ME_OP_2x3',
 '6_Portfolios_ME_OP_2x3_Wout_Div',
 '6_Portfolios_ME_OP_2x3_daily',
 '25_Portfolios_ME_OP_5x5',
 '25_Portf

In [3]:
# Daily 3 Factor
df = pdr.famafrench.FamaFrenchReader('F-F_Research_Data_Factors_daily')

In [4]:
data = df.read()

/var/folders/mr/nzdp38cj3cz3b0gcgyvws4c40000gp/T/ipykernel_57117/1535062722.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  data = df.read()


In [5]:
data

{0:             Mkt-RF   SMB   HML    RF
 Date                                
 2021-04-14   -0.34  0.79  1.41  0.00
 2021-04-15    1.06 -0.44 -1.19  0.00
 2021-04-16    0.27 -0.36  0.65  0.00
 2021-04-19   -0.72 -0.78  0.79  0.00
 2021-04-20   -0.93 -1.18 -1.57  0.00
 ...            ...   ...   ...   ...
 2026-02-23   -1.18 -0.30 -1.31  0.01
 2026-02-24    0.83  0.51 -0.66  0.01
 2026-02-25    0.79 -0.37  0.49  0.01
 2026-02-26   -0.47  0.63  0.32  0.01
 2026-02-27   -0.51 -0.44 -1.25  0.01
 
 [1225 rows x 4 columns],
 'DESCR': 'F-F Research Data Factors daily\n-------------------------------\n\nThis file was created by using the 202602 CRSP database. The Tbill return is the simple daily rate that, over the number of trading days compounds to 1-month TBill rate. The 1-month TBill rate data until 202405 are from Ibbotson Associates. Starting from 202406, the 1-month TBill rate is from ICE BofA US 1-Month Treasury Bill Index. Copyright 2026 Eugene F. Fama and Kenneth R. French\n\n  0 : 

In [6]:
data[0]

,Mkt-RF,SMB,HML,RF
Date,,,,
2021-04-14,-0.34,0.79,1.41,0.00
2021-04-15,1.06,-0.44,-1.19,0.00
2021-04-16,0.27,-0.36,0.65,0.00
2021-04-19,-0.72,-0.78,0.79,0.00
2021-04-20,-0.93,-1.18,-1.57,0.00
...,...,...,...,...
2026-02-23,-1.18,-0.30,-1.31,0.01
2026-02-24,0.83,0.51,-0.66,0.01
2026-02-25,0.79,-0.37,0.49,0.01


In [7]:
AAPL = yfinance.Ticker("NVDA")

In [8]:
AAPL.info

{'address1': '2788 San Tomas Expressway',
 'city': 'Santa Clara',
 'state': 'CA',
 'zip': '95051',
 'country': 'United States',
 'phone': '408 486 2000',
 'website': 'https://www.nvidia.com',
 'industry': 'Semiconductors',
 'industryKey': 'semiconductors',
 'industryDisp': 'Semiconductors',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization

In [9]:
AAPL.info['symbol']

'NVDA'

In [10]:
history = AAPL.history(period='5y')

In [11]:
history = history.reset_index()

In [12]:
history['Date'] = history['Date'].dt.date

In [13]:
history

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2021-04-14,15.585719,15.680979,15.189218,15.238594,385500000,0.0,0.0
1,2021-04-15,15.623125,16.173488,15.592202,16.096682,598480000,0.0,0.0
2,2021-04-16,16.012641,16.125108,15.825363,15.872495,335208000,0.0,0.0
3,2021-04-19,15.497440,15.767758,15.194703,15.323129,404420000,0.0,0.0
4,2021-04-20,15.284725,15.458039,14.925382,15.133108,334132000,0.0,0.0
...,...,...,...,...,...,...,...,...
1250,2026-04-07,175.729996,178.229996,173.660004,178.100006,132534900,0.0,0.0
1251,2026-04-08,184.500000,185.259995,180.300003,182.080002,147732700,0.0,0.0
1252,2026-04-09,181.839996,184.080002,180.619995,183.910004,116428500,0.0,0.0
1253,2026-04-10,184.309998,190.000000,184.300003,188.630005,160099500,0.0,0.0


In [14]:
history['Previous Close'] = history['Close'].shift(1)

In [15]:
history['Percentage Change'] = ((history['Close'] - history['Previous Close']) / history['Previous Close']) * 100

In [16]:
history = history.dropna()

In [17]:
condensed_history = history[['Date', 'Percentage Change']]

In [18]:
condensed_history

,Date,Percentage Change
1,2021-04-15,5.631015
2,2021-04-16,-1.392752
3,2021-04-19,-3.461119
4,2021-04-20,-1.240090
5,2021-04-21,1.247437
...,...,...
1250,2026-04-07,0.258954
1251,2026-04-08,2.234697
1252,2026-04-09,1.005054
1253,2026-04-10,2.566473


In [19]:
Fama_3_Factor = data[0].reset_index()

In [20]:
Fama_3_Factor['Date'] = Fama_3_Factor['Date'].dt.date

In [21]:
Fama_3_Factor

,Date,Mkt-RF,SMB,HML,RF
0,2021-04-14,-0.34,0.79,1.41,0.00
1,2021-04-15,1.06,-0.44,-1.19,0.00
2,2021-04-16,0.27,-0.36,0.65,0.00
3,2021-04-19,-0.72,-0.78,0.79,0.00
4,2021-04-20,-0.93,-1.18,-1.57,0.00
...,...,...,...,...,...
1220,2026-02-23,-1.18,-0.30,-1.31,0.01
1221,2026-02-24,0.83,0.51,-0.66,0.01
1222,2026-02-25,0.79,-0.37,0.49,0.01
1223,2026-02-26,-0.47,0.63,0.32,0.01


In [22]:
Combined = Fama_3_Factor.merge(condensed_history, on='Date', how='inner')

In [23]:
Combined

,Date,Mkt-RF,SMB,HML,RF,Percentage Change
0,2021-04-15,1.06,-0.44,-1.19,0.00,5.631015
1,2021-04-16,0.27,-0.36,0.65,0.00,-1.392752
2,2021-04-19,-0.72,-0.78,0.79,0.00,-3.461119
3,2021-04-20,-0.93,-1.18,-1.57,0.00,-1.240090
4,2021-04-21,1.10,1.38,0.12,0.00,1.247437
...,...,...,...,...,...,...
1219,2026-02-23,-1.18,-0.30,-1.31,0.01,0.911389
1220,2026-02-24,0.83,0.51,-0.66,0.01,0.678672
1221,2026-02-25,0.79,-0.37,0.49,0.01,1.405230
1222,2026-02-26,-0.47,0.63,0.32,0.01,-5.456124


In [24]:
Combined['Percent Change minus RF'] = (Combined['Percentage Change'] - (Combined['RF']))


In [25]:
Combined = Combined.drop(columns = 'Percentage Change')

In [26]:
Combined.columns

Index(['Date', 'Mkt-RF', 'SMB', 'HML', 'RF', 'Percent Change minus RF'], dtype='object')

In [27]:
tscv = TimeSeriesSplit(n_splits=4, test_size=252, gap=5)

X = Combined[['Mkt-RF', 'SMB', 'HML']]
y = Combined[['Percent Change minus RF']]

split_data = []

for i, (train_index, test_index) in enumerate(tscv.split(Combined)):
    train_dates = Combined.iloc[train_index]['Date']
    test_dates = Combined.iloc[test_index]['Date']

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Initialize and Fit
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred

    # Check your Factor Loadings (Betas) for this fold
    print(f"Mkt-RF Beta: {model.coef_[0]}")
    print(f"Alpha (Intercept): {model.intercept_}")
    print(f"R2 score: {model.score(X_test, y_test)}")
    # If the average residual is positive, the stock outperformed the model's expectations
    print(f"Mean Prediction Error: {residuals.mean()}")
    X_train_with_const = sm.add_constant(X_train)

    mse_val = mean_squared_error(y_test, y_pred)
    rmse_val = np.sqrt(mse_val)

    # 2. Fit the OLS (Ordinary Least Squares) model
    ols_model = sm.OLS(y_train, X_train_with_const).fit()

    # 3. Access the p-values
    p_values = ols_model.pvalues
    print(f"P-values for this fold:\n{p_values}")

    # 4. View the full academic summary
    print(ols_model.summary())
    
    # 4. Store metrics in a dictionary
    fold_result = {
        'Fold': i + 1,
        'Period_Start': test_dates.min(),
        'Period_End': test_dates.max(),
        # Accuracy Metrics
        'In_Sample_R2': ols_model.rsquared,
        'Out_of_Sample_R2': model.score(X_test, y_test),
        'RMSE': rmse_val,
        'Mean Prediciton Error': residuals.mean(),
        # Coefficients (Betas)
        'Alpha': ols_model.params['const'],
        'Mkt_Beta': ols_model.params['Mkt-RF'],
        'SMB_Beta': ols_model.params['SMB'],
        'HML_Beta': ols_model.params['HML'],
        # P-Values (Is it significant?)
        'Alpha_P': ols_model.pvalues['const'],
        'Mkt_P': ols_model.pvalues['Mkt-RF'],
        'SMB_P': ols_model.pvalues['SMB'],
        'HML_P': ols_model.pvalues['HML']
    }
    
    split_data.append(fold_result)


Mkt-RF Beta: [ 1.95148438 -0.14069482 -0.91334307]
Alpha (Intercept): [0.29910372]
R2 score: 0.7313143229655948
Mean Prediction Error: Percent Change minus RF   -0.211606
dtype: float64
P-values for this fold:
const     2.428251e-02
Mkt-RF    2.706357e-26
SMB       4.714399e-01
HML       1.669461e-12
dtype: float64
                               OLS Regression Results                              
Dep. Variable:     Percent Change minus RF   R-squared:                       0.584
Model:                                 OLS   Adj. R-squared:                  0.578
Method:                      Least Squares   F-statistic:                     96.76
Date:                     Mon, 13 Apr 2026   Prob (F-statistic):           3.55e-39
Time:                             15:23:12   Log-Likelihood:                -432.19
No. Observations:                      211   AIC:                             872.4
Df Residuals:                          207   BIC:                             885.8
Df Model:  

In [28]:
# Convert to DataFrame for the Stability Report
stability_df = pd.DataFrame(split_data)

In [29]:
stability_df

,Fold,Period_Start,Period_End,In_Sample_R2,Out_of_Sample_R2,RMSE,Mean Prediciton Error,Alpha,Mkt_Beta,SMB_Beta,HML_Beta,Alpha_P,Mkt_P,SMB_P,HML_P
0,1,2022-02-22,2023-02-22,0.583740,0.731314,2.037606,Percent Change minus RF -0.211606 dtype: flo...,0.299104,1.951484,-0.140695,-0.913343,0.024283,2.706357e-26,0.471440,1.669461e-12
1,2,2023-02-23,2024-02-23,0.693583,0.335532,2.480088,Percent Change minus RF 0.167973 dtype: flo...,0.195963,1.840935,0.015548,-0.833653,0.033045,1.646784e-78,0.916738,1.361555e-17
2,3,2024-02-26,2025-02-26,0.599360,0.493362,2.439094,Percent Change minus RF -0.086509 dtype: flo...,0.233688,1.876239,-0.265937,-0.898798,0.003235,9.985005e-97,0.027520,4.706592e-25
3,4,2025-02-27,2026-02-27,0.572409,0.655838,1.641227,Percent Change minus RF -0.138313 dtype: flo...,0.208413,1.985925,-0.448478,-1.032325,0.003690,2.992400e-118,0.000036,1.448763e-35


In [30]:

X_full = Combined[['Mkt-RF', 'SMB', 'HML']]
y_full = Combined['Percent Change minus RF']

X_full_const = sm.add_constant(X_full)


final_model = sm.OLS(y_full, X_full_const).fit()

# 4. Display the results for your dashboard
print(final_model.summary())

# Extracting values from the final_model object
results_summary = {
    'Ticker': AAPL.info['symbol'],
    'Name': AAPL.info['shortName'],
    'Industry': AAPL.info['industry'],
    'Summary': AAPL.info['longBusinessSummary'],
    'Current_Alpha': final_model.params['const'],
    'Market_Beta': final_model.params['Mkt-RF'],
    'Size_Beta': final_model.params['SMB'],
    'Value_Beta': final_model.params['HML'],
    'Alpha_PValue': final_model.pvalues['const'],
    'Overall_Fit': final_model.rsquared_adj,
    'Lower_Alpha_Bound': final_model.conf_int().loc['const', 0],
    'Upper_Alpha_Bound': final_model.conf_int().loc['const', 1]
}

                               OLS Regression Results                              
Dep. Variable:     Percent Change minus RF   R-squared:                       0.587
Model:                                 OLS   Adj. R-squared:                  0.586
Method:                      Least Squares   F-statistic:                     577.1
Date:                     Mon, 13 Apr 2026   Prob (F-statistic):          2.03e-233
Time:                             15:23:12   Log-Likelihood:                -2648.7
No. Observations:                     1224   AIC:                             5305.
Df Residuals:                         1220   BIC:                             5326.
Df Model:                                3                                         
Covariance Type:                 nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
co

In [31]:
# Daily 3 Factor
fivef_df = pdr.famafrench.FamaFrenchReader('F-F_Research_Data_5_Factors_2x3_daily')

In [32]:
fivef_df[0]

TypeError: 'FamaFrenchReader' object is not subscriptable